# Experiment 02 on Colab: claim splitting × evidence windows

Runs `experiments/02_claims.py`. Llama 3.1 (Ollama) splits answers into claims, then DeBERTa
checks them. Four configurations, all scored against the same labels:

| | evidence = whole passage | evidence = 3-sentence windows |
|---|---|---|
| **sentences** | baseline rerun (with fixes) | evidence fix only |
| **claims (Llama)** | claim splitting only | both |

**Runtime → Change runtime type → T4 GPU.** Everything saves to Google Drive as it goes; after a
disconnect, run the setup cells again, then the step you were on. It resumes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Settings

In [ ]:
REPO = "https://github.com/vedantrajeshirke/agentic-hallucination-detection.git"
REF = "master"            # pin to a commit hash for a reported run
DEV_PER_TASK = "100"      # dev answers per task for tuning ("" = full dev split, 1,506 answers)
WORKERS = 4               # parallel Llama requests
OUT = "/content/drive/MyDrive/agentic-hallucination/results/02_claims"

## Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(OUT, exist_ok=True)

In [ ]:
%cd /content
!test -d repo || git clone -q $REPO repo
%cd /content/repo
!git fetch -q origin && git checkout -q $REF && git log -1 --oneline

In [ ]:
!pip -q install uv
!uv sync -q --python 3.12
!uv run python -c "import torch; print(torch.__version__, 'CUDA:', torch.cuda.is_available())"

If that printed `CUDA: False`, use the fallback cell in `colab/01_sentence_baseline.ipynb` (and add `--no-sync` after `uv run` below).

In [ ]:
# Ollama for Llama 3.1. Expected digest (author's machine): 46e0c10c039e
# zstd is needed by the installer; its output is shown so a failure is visible.
import subprocess, time, os, shutil
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!test -x /usr/local/bin/ollama || (curl -fsSL https://ollama.com/install.sh | sh)
OLLAMA = shutil.which("ollama") or "/usr/local/bin/ollama"
print("ollama found at:", OLLAMA, "| exists:", os.path.exists(OLLAMA))
env = dict(os.environ, OLLAMA_NUM_PARALLEL=str(WORKERS), OLLAMA_HOST="127.0.0.1:11434")
subprocess.Popen([OLLAMA, "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!{OLLAMA} pull llama3.1:8b > /dev/null 2>&1; {OLLAMA} list

## Step 1: dev (for tuning)

Decompose, then score all four configurations. Each cell resumes if interrupted; finished steps print `0 to go` and return in seconds, so after a disconnect it is safe to run them again.


In [ ]:
per_task = f"--per-task {DEV_PER_TASK}" if DEV_PER_TASK else ""
!uv run python experiments/02_claims.py decompose --split dev $per_task --workers $WORKERS --out-dir "$OUT"

In [ ]:
for claims in ["sentences", "llm"]:
    for evidence in ["passage", "s3"]:
        print(f"\n=== dev: claims={claims}, evidence={evidence}")
        !uv run python experiments/02_claims.py run --split dev $per_task --claims $claims --evidence $evidence --fp16 --batch-size 32 --out-dir "$OUT"

## Step 2: test (2,700 answers)

In [ ]:
!uv run python experiments/02_claims.py decompose --split test --workers $WORKERS --out-dir "$OUT"

In [ ]:
for claims in ["sentences", "llm"]:
    for evidence in ["passage", "s3"]:
        print(f"\n=== test: claims={claims}, evidence={evidence}")
        !uv run python experiments/02_claims.py run --split test --claims $claims --evidence $evidence --fp16 --batch-size 32 --out-dir "$OUT"

## Step 3: evaluate (tune on dev, report on test)

In [ ]:
dev_tag = f"dev_n{DEV_PER_TASK}" if DEV_PER_TASK else "dev"
for claims in ["sentences", "llm"]:
    for evidence in ["passage", "s3"]:
        tune = f"{OUT}/scores_{dev_tag}_{claims}_{evidence}.jsonl"
        report = f"{OUT}/scores_test_{claims}_{evidence}.jsonl"
        print(f"\n\n######## claims={claims}, evidence={evidence}")
        !uv run python experiments/02_claims.py evaluate --tune "$tune" --report "$report"